In [2]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [4]:
model_id = "ScorpieCur/mistral-7b-instruct-v0.3-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

In [5]:
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=200
)

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
def build_prompt(user_query):
    return f"Act like a friendly medical assistant and provide general health info only. Question: {user_query}"

In [7]:
def safety_filter(query):
    risky = ["prescription", "dosage", "how much", "take medicine", "mg", "tablet"]
    for word in risky:
        if word in query.lower():
            return " I cannot provide medical prescriptions or dosages. Please consult a healthcare professional."
    return None

In [8]:
def health_chatbot(user_query):
    warning = safety_filter(user_query)
    if warning:
        return warning

    prompt = build_prompt(user_query)
    response = chatbot(prompt, max_new_tokens=150)
    return response[0]['generated_text']

In [9]:
print(health_chatbot("What causes a sore throat?"))
print(health_chatbot("Is paracetamol safe for children?"))
print(health_chatbot("What are symptoms of flu?"))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Act like a friendly medical assistant and provide general health info only. Question: What causes a sore throat? Answers: A sore throat can be caused by several factors, such as:

1. Viral Infections: The common cold, flu, or other viral illnesses like mononucleosis can cause a sore throat.

2. Bacterial Infections: Streptococcus pyogenes, a type of bacterium, can cause strep throat, which is a more severe type of sore throat.

3. Allergies: Allergies to pollen, dust, or pets can cause a sore throat, especially in the back of the throat.

4. Irritants: Irritants such as smoke, dry air, or acidic food can also cause a


Both `max_new_tokens` (=150) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Act like a friendly medical assistant and provide general health info only. Question: Is paracetamol safe for children?

Answer: Paracetamol is commonly used to relieve pain and reduce fever in children, but it's important to follow the recommended dosage on the package or as directed by a healthcare provider. Overdose can be harmful, so always carefully measure doses using a spoon or oral syringe provided with the medication. If you have concerns about your child's health or if they show signs of an allergic reaction, seek immediate medical advice. Always consult a healthcare professional for personalized advice.
Act like a friendly medical assistant and provide general health info only. Question: What are symptoms of flu?

Answer: The flu, also known as influenza, is a contagious respiratory illness that can cause a wide range of symptoms. Here are some common symptoms:

1. Fever: This is one of the most common symptoms of the flu. The fever can range from mild to high, and it usuall

In [11]:
while True:
    user_input = input("Ask a health question (type 'exit' to stop): ")

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    response = health_chatbot(user_input)
    print("Bot:", response)

Ask a health question (type 'exit' to stop): What causes fever?


Both `max_new_tokens` (=150) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Act like a friendly medical assistant and provide general health info only. Question: What causes fever?

Answer: A fever, or elevated body temperature, is typically caused by your body's response to an infection or illness. The immune system releases chemicals called pyrogens to increase body temperature as a way to combat the invading organism. Other less common causes include certain medications, physical exertion, and immunizations. However, it's important to note that not all fevers are caused by infections. In some cases, a fever can be a side effect of a condition or medication. If you have a fever, it's always a good idea to consult with a healthcare professional for proper diagnosis and treatment.
Ask a health question (type 'exit' to stop): exit
Chat ended.


In [12]:
from huggingface_hub import snapshot_download

model_id = "ScorpieCur/mistral-7b-instruct-v0.3-bnb-4bit"

local_dir = snapshot_download(
    repo_id=model_id,
    local_dir="./mistral_model",
    local_dir_use_symlinks=False
)

print("Saved at:", local_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Saved at: /content/mistral_model


In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained("./mistral_model")
model = AutoModelForCausalLM.from_pretrained("./mistral_model", device_map="auto")

chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [14]:
print(chatbot("What causes fever?")[0]['generated_text'])

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What causes fever?

Fever is a sign of the body’s immune response to infection or inflammation. When the body is under attack by a foreign substance, such as a virus or bacteria, it triggers a response that raises body temperature to help fight off the intruder. This increase in temperature can help kill off the invading organism by making it less able to multiply and survive at higher temperatures.

What is a normal temperature?

The average normal body temperature is around 98.6°F (37°C), but it can vary slightly between individuals, and can be influenced by factors such as exercise, stress, and illness. A temperature of 100.4°F (3
